## Set up Data

#### Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Unzip the Images

In [ ]:
import zipfile, os

CHALLENGE_DIR = "/content/drive/MyDrive/AML"
ZIP_PATH      = os.path.join(CHALLENGE_DIR, "dataset.zip")
IMAGES_DIR    = "/content/images"   # extracted here (fast local SSD, not Drive)

if not os.path.exists(IMAGES_DIR):
    print("Extracting images...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(IMAGES_DIR)
    print(f"Done — extracted to {IMAGES_DIR}")
else:
    print(f"Already extracted: {IMAGES_DIR}")

# Peek at what's inside
sample_files = os.listdir(os.path.join(IMAGES_DIR,"images"))[:5]
print(f"Total images : {len(os.listdir(os.path.join(IMAGES_DIR,"images")))}")
print(f"Sample names : {sample_files}")
IMAGES_DIR = os.path.join(IMAGES_DIR,"images")

#### Load the Training Data

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(CHALLENGE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(CHALLENGE_DIR, "test.csv"))

# Map image_id → full path
def img_path(image_id):
    # Try common extensions; adjust if your images have a fixed extension
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(p):
            return p
    return None

train_df["path"] = train_df["image_id"].apply(img_path)
test_df["path"]  = test_df["image_id"].apply(img_path)

print(f"Train rows   : {len(train_df)}")
print(f"Test  rows   : {len(test_df)}")
print("\nLabel distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))


## DinoV2 + ConvNextV2

### Extract features

In [ ]:
!pip install torch timm

In [ ]:
import torch
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
dinov2 = dinov2.to(device)

convnext = timm.create_model('convnextv2_base', pretrained=True, num_classes=0)
convnext = convnext.to(device)

dinov2.eval()
convnext.eval()

data_config = timm.data.resolve_data_config(convnext.pretrained_cfg)
preprocess = timm.data.create_transform(**data_config, is_training=False)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class AIDetectionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        label = self.dataframe.iloc[idx]['ground_truth']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def extract_features(df, batch_size=16):
    dataset = AIDetectionDataset(train_df, transform=preprocess)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    features_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting Features"):
            images = images.to(device)

            f_dino = dinov2(images)
            f_conv = convnext(images)

            # Concatenate features along the channel dimension
            f_combined = torch.cat((f_dino, f_conv), dim=1)

            features_list.append(f_combined.cpu().numpy())
            labels_list.append(labels.numpy())

    return np.vstack(features_list), np.concatenate(labels_list)

X, y = extract_features(train_df)

#### Save features

In [ ]:
FEATURES_PATH = os.path.join(CHALLENGE_DIR, "features/dino+conv_extracted_features.npy")
LABELS_PATH = os.path.join(CHALLENGE_DIR, "features/dino+conv_extracted_labels.npy")

np.save(FEATURES_PATH, X)
np.save(LABELS_PATH, y)

print(f"Features saved to: {FEATURES_PATH}")
print(f"Labels saved to: {LABELS_PATH}")

### Load Features

In [ ]:
import numpy as np

FEATURES_PATH = os.path.join(CHALLENGE_DIR, "features/dino+conv_extracted_features.npy")
LABELS_PATH = os.path.join(CHALLENGE_DIR, "features/dino+conv_extracted_labels.npy")

X = np.load(FEATURES_PATH)
y = np.load(LABELS_PATH)

## Classifier

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, f1_score
from sklearn.svm import SVC

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
param_grid = {
    # 'C': range(5, 15)
    'kernel': ["linear", "poly", "rbf", "sigmoid"]
}

gridSVM = GridSearchCV(SVC(), param_grid, scoring='f1')
gridSVM.fit(X_train, y_train)

print(gridSVM.best_params_, gridSVM.best_score_)

In [ ]:
classifierSVM = SVC()
classifierSVM.fit(X_train, y_train)

In [ ]:
y_predSVM = classifierSVM.predict(X_val)

print("\nSVM Classification Report:")
print(classification_report(y_val, y_predSVM, target_names=["Authentic", "AI-Generated"]))
print(f"F1: {f1_score(y_val, y_predSVM):.3f}")

In [ ]:
scores = cross_val_score(classifierSVM, X, y, scoring="f1")
print(f"SVM: Mean: {scores.mean():.3f}, Std: {scores.std():.3f}")